# Weather Utility — Example Notebook

This notebook demonstrates how to call the public functions in `utils/weather_utils.py` and inspect their outputs.

## Public functions covered
- `load_weather_config`
- `build_weather_for_household_calendar`
- `fetch_raw_historical_weather`
- `build_full_weather_15min`
- `add_derived_weather_features`

In [ ]:
import sys
from pathlib import Path

# Ensure notebook imports resolve from project root and src/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"

for p in (PROJECT_ROOT, SRC_DIR):
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import matplotlib.pyplot as plt
import pandas as pd

# Make sure project root is importable when notebook runs from notebooks/
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from utils.weather_utils import (
    add_derived_weather_features,
    build_full_weather_15min,
    build_weather_for_household_calendar,
    fetch_raw_historical_weather,
    load_weather_config,
)

print(f"Project root: {root}")

## 1) Load config and define a sample calendar

We build a small 2-day 15-minute UTC calendar to keep this example fast.

In [ ]:
config = load_weather_config("user_config.json")
print({k: config[k] for k in ["lat", "lon"]})

sample_start = pd.Timestamp("2025-01-15 00:00:00", tz="UTC")
sample_end = pd.Timestamp("2025-01-16 23:45:00", tz="UTC")
calendar_index = pd.date_range(sample_start, sample_end, freq="15min")
household_calendar = pd.DataFrame({"utc_timestamp": calendar_index})

print(f"Calendar rows: {len(household_calendar)}")
household_calendar.head(3)

## 2) End-to-end call: `build_weather_for_household_calendar`

This runs the full pipeline (fetch raw hourly weather + interpolate to 15-minute table).

In [ ]:
weather_15min = build_weather_for_household_calendar(
    household_timestamps=household_calendar,
    config_path="user_config.json",
    include_derived=False,
)

print(f"Rows: {len(weather_15min)}")
print(f"Columns: {len(weather_15min.columns)}")
weather_15min.head(5)

## 3) Individual function calls

Below we call the lower-level helpers directly to show what each one returns.

In [ ]:
raw_weather = fetch_raw_historical_weather(
    household_calendar=household_calendar,
    config_path="user_config.json",
)

print(f"Raw hourly rows: {len(raw_weather)}")
raw_weather[["time", "temperature_2m", "cloud_cover", "source"]].head(5)

In [ ]:
full_15min = build_full_weather_15min(
    raw_weather_df=raw_weather,
    household_calendar=household_calendar,
)

print(f"Interpolated 15-min rows: {len(full_15min)}")
full_15min[["utc_timestamp", "temperature_2m", "wind_speed_10m", "source"]].head(5)

In [ ]:
derived = add_derived_weather_features(full_15min)

derived_cols = [
    "heating_degree_18c",
    "cooling_degree_22c",
    "is_raining",
    "is_snowing",
    "is_dark",
    "wind_u_10m",
    "wind_v_10m",
]

print("Derived columns added:")
print(derived_cols)
derived[["utc_timestamp", *derived_cols]].head(5)

## 4) Short EDA of the 15-minute weather output

In [ ]:
eda_df = weather_15min.copy()

print("Shape:", eda_df.shape)
print()
print("dtypes (first 10):")
print(eda_df.dtypes.head(10))
print()
eda_df[["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "precipitation"]].describe()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.plot(eda_df["utc_timestamp"], eda_df["temperature_2m"], color="#ef4444", label="temperature_2m")
ax1.set_xlabel("UTC timestamp")
ax1.set_ylabel("Temperature (°C)", color="#ef4444")
ax1.tick_params(axis="y", labelcolor="#ef4444")

ax2 = ax1.twinx()
ax2.plot(eda_df["utc_timestamp"], eda_df["relative_humidity_2m"], color="#2563eb", alpha=0.75, label="relative_humidity_2m")
ax2.set_ylabel("Relative humidity (%)", color="#2563eb")
ax2.tick_params(axis="y", labelcolor="#2563eb")

ax1.set_title("Weather utility output: temperature and humidity (15-min)")
fig.tight_layout()
plt.show()